In [1]:
import polars as pl

from mybookrec import DATA_DIR

In [2]:
# Schemas contracts for transformed data.
BOOKS_SCHEMA = pl.Struct(
    {
        "book_id": pl.Int64,
        "num_pages": pl.Int64,
        "average_rating": pl.Float64,
        "ratings_count": pl.Int64,
        "description": pl.String,
        "title": pl.String,
        "language_code": pl.String,
    }
)

INTERACTIONS_SCHEMA = pl.Struct(
    {
        "user_id": pl.Int64,
        "book_id": pl.Int64,
        "rating": pl.Float64
    }
)

MY_BOOKS_SCHEMA = pl.Struct(
    {
        "book_id": pl.Int64,
        "my_rating": pl.Float64,
        "average_rating": pl.Float64,
    }
)

GENRES_SCHEMA = pl.Struct(
    {
        "book_id": pl.Int64,
        "genres": pl.Object

    }
)

In [5]:
# Load raw book data and cast to the defined schema, while applying necessary transformations and filters
books_load = pl.scan_ndjson(
    DATA_DIR / "raw" / "goodreads_books.json.gz",
    low_memory=True
).select(
    pl.col("book_id"),
    pl.col("num_pages").cast(pl.Int64, strict=False), # Cast num_pages to int, allowing for nulls if conversion fails.
    pl.col("is_ebook") == True, # Produce boolean column indicating if the book is an ebook or not.
    pl.col("average_rating").cast(pl.Float64, strict=False), # Cast average_rating to float, allowing for nulls if conversion fails.
    pl.col("ratings_count").cast(pl.Int64, strict=False), # Cast ratings_count to int, allowing for nulls if conversion fails.
    pl.col("description"),
    pl.col("title"),
    pl.col("language_code")
).filter(
    pl.col("ratings_count") > 5, # Filter out books with no ratings to avoid skewing the model with unrated books.
    pl.col("language_code") == "en", # Filter out non-English books to focus on the majority of the data and avoid language-related issues in modeling.
)

# Load book genres data to join to the main books dataframe, while selecting only relevant columns and applying necessary transformations.
genres_load = pl.scan_ndjson(
    DATA_DIR / "raw" / "goodreads_book_genres_initial.json.gz",
    low_memory=True
).select(
    pl.col("book_id"),
    pl.col("genres")
)

# Load user interactions data and cast to the defined schema, while applying necessary transformations and filters
interactions_load = pl.scan_ndjson(
    DATA_DIR / "raw" / "goodreads_interactions_dedup.json.gz",
    low_memory=True
).select(
    pl.col("user_id"),
    pl.col("book_id"),
    pl.col("rating").cast(pl.Float64, strict=False) # Cast rating to float, allowing for nulls if conversion fails.
).filter(
    pl.col("rating") > 0 # Filter out interactions with no rating to focus on explicit feedback and avoid skewing the model with implicit interactions.
)

# Load my data and filter to only include books that I have rated, while selecting only relevant columns and applying necessary transformations.
my_books_load = pl.read_csv(
    DATA_DIR / "raw" / "goodreads_library_export.csv"
).select(
    pl.col("Book Id").alias("book_id").cast(pl.Int64, strict=False), # Cast book_id to int, allowing for nulls if conversion fails.
    pl.col("My Rating").alias("my_rating").cast(pl.Float64, strict=False) # Cast my_rating to float, allowing for nulls if conversion fails.
).filter(
    pl.col("my_rating") > 0 # Filter out books that I have not rated to focus on explicit feedback and avoid skewing the model with implicit interactions.
).write_csv(
    DATA_DIR / "transformed" / "my_books.csv"
)

In [6]:
# Inner join books and genres so we can drop books without genre information, which is important for content-based filtering and to avoid missing data issues in modeling. then save to parquet for later use.
books_with_genres = books_load.join(
    genres_load,
    on="book_id",
    how="inner"
).sink_parquet(
    DATA_DIR / "transformed" / "books_with_genres.parquet"
)

# Inner join books and interactions to focus on books that have user interactions, which is important for collaborative filtering and to avoid skewing the model with books that have no user feedback. then save to parquet for later use.
books_with_interactions = books_load.join(
    interactions_load,
    on="book_id",
    how="inner"
).sink_parquet(
    DATA_DIR / "transformed" / "books_with_interactions.parquet"
)